# A Look At My Past 100 MCSR Ranked Matches 

## Introduction
For the course project, I grabbed data from https://docs.mcsrranked.com/, which contains the past 100 ranked games I've played in MCSR Ranked, a Minecraft mod that lets players compete against each other to see who is faster in beating the seed. My focus is to visualize the amount of the four different seed types and bastion types I've gotten throughout my $100$ ranked matches. For machine learning, I'll be using `LinearRegression()` to create a prediction line based on my completion times. 

## Importing modules, loading and cleaning data

In [1]:
import pandas as pd
import json
import requests
import altair as alt
import numpy as np
from pandas import json_normalize
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
matches = requests.get("https://api.mcsrranked.com/users/swibsub/matches/", params = {"type": 2, "count": 100})
data = matches.json()

In [3]:
df = pd.json_normalize(data["data"])
df.head()

Dropping columns from the data that are irrelevant for this assignment.

In [4]:
df = df.drop(["id", "gameMode", "spectators", "decayed", "vod", "changes", "beginner", "botSource", "tag", "seed.variations", "rank.season", "rank.allTime"], axis=1)

In [5]:
df.info()

There are no columns with missing data besides `result.uuid`. Although `result.uuid` counts matches that are drawn as `None`, so there is no need to drop the rows that have this "missing data".

In [6]:
df.isna().any().any()

Since the column, `date` uses the epoch timestamp, I will be creating a new column `dates` with the dates converted. Additionally, since `result.time` is in milliseconds, I will also be converting those values into minutes by dividing each value by $60000$.

In [7]:
from datetime import datetime as dt

In [8]:
df["dates"] = df["date"].apply(dt.fromtimestamp)

In [9]:
df["result.time"] = df["result.time"] / 60000

In [10]:
df.head()

Next, I will be creating a `sub_df` of all the of my victorious matches (that aren't forfeits) which will be used for machine learning later.

In [11]:
sub_df = df[df["forfeited"] == False]

In [12]:
sub_df = sub_df[sub_df["result.uuid"] == "385770ff8d89461e8cc71760c7a60a9e"]

sub_df.info()

What remains after cleaning up the data are $21$ rows in the `sub_df`. I will be creating a new dataframe called `df_times` which will be ordered for my latest to earliest completions.

In [13]:
df_times = pd.DataFrame({
    "x": range(21),
    "y": sub_df["result.time"].iloc[::-1].reset_index(drop = True)
})

df_times.head()

## Visaulization of the Data

In [14]:
c = alt.Chart(df).mark_bar().encode(
    x = "seedType:N",
    y = "count()",
    color = alt.Color("seedType:N", scale = alt.Scale(range = ["coral", "gold", "orangered", "royalblue",])),
    tooltip = "count()"
)

c

This bar graph shows the amount of each seed type I've gotten from the past 100 matches (where there are only $5$ to pick from). In MCSR Ranked, any elo above $1200$ has a $20\%$ chance of playing each seed type. Although, this graph shows that I've been getting a significant amount of Desert Temple seeds compared to the other four I could've possibly gotten. This is from a somewhat low sample size (limited to 100), so if I was able to include more of my matches, the counts for each seed type should be more balanced. 

In [15]:
c2 = alt.Chart(df).mark_bar().encode(
    x = "bastionType:N",
    y = "count()",
    color = alt.Color("bastionType:N", scale = alt.Scale(range = ["darkorange", "crimson", "darkred", "goldenrod"])),
    tooltip = "count()"
)

c2

This bar graph shows the amount of each the four possible bastions I've played spanning the 100 ranked matches. In Minecraft, there is a $25\%$ chance of that the bastion type can either be Bridge, Housing, Stables or Treasure. Although, this graph shows that I've been getting a significant amount of Bridge and Housing bastions compared to the other two I could've possibly gotten. This is from a somewhat low sample size (limited to 100), so if I was able to include more of my matches, the counts for each bastion type should be more balanced. 

## Machine Learning

Down below is a chart of the times of my $35$ completions throughout my past $100$ ranked matches.

In [16]:
r = alt.Chart(df_times).mark_trail(color = "green").encode(
    x = alt.X("x", title = "Number of matches", scale = alt.Scale(zero = False)),
    y = alt.Y("y", title = "Completion times (in minutes)",scale = alt.Scale(zero = False)),
    tooltip = "y"
)

r

Next, I'll be using `LinearRegression()`, which will create a line that is best fit.

In [17]:
reg = LinearRegression()

reg.fit(df_times[["x"]], df_times["y"])

In [18]:
reg.coef_ 

In this context, this is the slope $m$ in the linear equation $y=mx+b$.

In [19]:
reg.intercept_

Additionally, this is the y-intercept $b$ in the linear equation $y=mx+b$.

With the slope and intercept, I'll now be creating a dataframe that predicts my completion times. 

In [35]:
x = np.linspace(0,34, 21)
y_pred = -0.03931752 * x + 12.436395952380952

In [37]:
df_pred = pd.DataFrame({
    "x": range(21),
    "y": y_pred
})

df_pred.head()

Now, I will visualize it using Altair below:

In [39]:
p = alt.Chart(df_pred).mark_trail(color = "red").encode(
    x = alt.X("x", title = "Number of matches", scale = alt.Scale(zero = False)),
    y = alt.Y("y", title = "Prediction of completion times (in minutes)",scale = alt.Scale(zero = False)),
    tooltip = "y"
)

p

Combining both charts, we'll see that the prediction line (which is red), has a negative slope, implying that my completions times have steadily been getting lower.

In [41]:
r + p

To see how accurate the prediction line is to the actual graph, I'll be calculating the mean square error:

In [42]:
mean_squared_error(df_times["y"],df_pred["y"])

Based on the mean_squared_eror, the true chart of my $21$ completions and predicted chart of my $21$ completions have a squared difference of $\approx3.01$.

# References
- Altair Library: https://altair-viz.github.io/
- MCSR Ranked API Website (Source of Dataset): https://docs.mcsrranked.com/#objects-matchinfo
- JSON Documentation: https://docs.python.org/3/library/json.html
- Numpy Library: https://numpy.org/doc/
- Pandas Library: https://pandas.pydata.org/docs/
- Requests Library: https://requests.readthedocs.io/en/latest/
- Scikit-learn Library: https://scikit-learn.org/stable/user_guide.html

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=2f32e854-b602-4e28-a67c-ac0655c9cdcf' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>